In [ ]:
for m in client.models.list():
    if "flash" in m.name.lower() or "gemini" in m.name.lower():
        print(m.name)

models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.0-flash
models/gemini-2.0-flash-001
models/gemini-2.0-flash-lite-001
models/gemini-2.0-flash-lite
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3-pro-preview
models/gemini-3-flash-preview
models/gemini-3.1-pro-preview
models/gemini-3.1-pro-preview-customtools
models/gemini-3.1-flash-lite-preview
models/gemini-3-pro-image-preview
models/gemini-3.1-flash-image-preview
models/gemini-3.1-flash-tts-preview
models/gemini-robotics-er-1.5-preview
models/gemini-robotics-er-1.6-preview
models/gemini-2.5-computer-use-preview-10-2025
models/gemini-embedding-001
models/gemini-embedding-2-preview
models/gemini-embedding-2
models/gemini-2.5-flash-native-audio-latest
models/gemini-2.5-flash-native-audio-preview-09-2025
models/gemini-2.5-flash-native-audio-preview

In [ ]:
# ============================================================================
# CELL 1 — Imports & Configuration
# ============================================================================
!pip install google-genai -q

import json, re, time, logging
import pandas as pd
from google import genai
from google.genai import types

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)


GEMINI_API_KEY = "API_Key"


client         = genai.Client(api_key=GEMINI_API_KEY)

GEMINI_MODELS = [
    "gemini-3.1-flash-lite-preview",
]

BASE_PATH  = "BASE_PATH"
INPUT_PATH = f"{BASE_PATH}/Input_Data_GEC.xlsx"

OUTPUT_PATHS = {
    "gemini-3.1-flash-lite-preview": f"{BASE_PATH}/Gemini_31FlashLitePreview_Responses_Data_GEC.xlsx",
}

SYSTEM_PROMPT = """You are an expert in Odia (ଓଡ଼ିଆ) linguistics specializing in Grammatical Error Detection (GED) and Grammatical Error Correction (GEC). Analyze the given text and identify all linguistic errors with high precision.

## Categories (apply in this priority order — assign only the highest-priority matching category per span)

1. Script Normalization — Unicode-level encoding errors where the character sequence is wrong even if the rendering looks similar to the correct form. Includes: nukta misplacement relative to a base consonant, incorrect virama usage in conjunct formation, vowel sign decomposition errors where two combining marks are used instead of a single precomposed one, and ZWNJ/ZWJ presence or absence errors affecting ligature formation or consonant cluster boundaries. Also includes inappropriate consonant conjunct or cluster formation caused by the absence of a ZWNJ, where two adjacent consonants incorrectly merge into a ligature instead of remaining as distinct units.

2. Spelling & Typographical Errors — Unicode encoding is correct but the wrong characters are used. Includes: confusion between short and long vowel signs, substitution among phonetically similar consonants, missing or extra characters within a word, and incorrect word boundaries (two words merged or one word split). [*Note: if the error is in how characters are encoded rather than which characters are chosen, classify as Script Normalization instead.]

3. Grammatical Errors — Morphosyntactic errors where the script and spelling are correct. Includes: wrong verb tense, aspect, or inflectional form; agreement failure; incorrect postpositional case marker; wrong conjunction for the syntactic context; incorrect voice; light verb misuse; word order violations; wrong copular constructions; and missing punctuation that is grammatically required.

4. Code-Mixing / Wrong Language — Odia text containing elements from a different language or script. Includes: Roman-script words or abbreviations where an Odia equivalent exists, non-Odia numeral systems where Odia digits are standard, characters from another Indic script embedded in Odia text, and loanword phrases where a standard Odia term is available.

5. Correct Sentence / No Errors — Return this as a single entry if no errors are found.

## Rules
- Priority: If a span qualifies under multiple categories, assign only the highest-priority matching category (1 is highest).
- Single error assumption: Each input sentence contains exactly one error, which may span a single character, a word, or a longer phrase (e.g., a word order violation). Identify the single best error span and assign it to exactly one category. If multiple categories seem applicable, rank them by the priority order above and assign the highest-ranking one.
- Span overlap: Report the one erroneous span only. Do not report sub-spans of the same error separately.
- Correction scope: Make minimum necessary edits only — do not reorder, paraphrase, or add content beyond what fixing the identified error requires.
- Binary flag: Set "has_errors" to true if an error is found, and false if the sentence is correct. When false, "errors" must be an empty array and "corrected_sentence" must be identical to the input.

## Output
Return ONLY valid JSON. No markdown, no text outside the JSON.

{
  "has_errors": <true | false>,
  "errors": [
    {
      "error_span": "<exact erroneous text>",
      "category": "<one of the five categories>",
      "description": "<technical explanation>"
    }
  ],
  "corrected_sentence": "<fully corrected sentence, or identical to input if no errors>"
}"""

In [ ]:
# ============================================================================
# CELL 2 — API Call & Response Parsing
# ============================================================================
def parse_response(text: str):
    text = re.sub(r'^```json\s*|^```\s*|\s*```$', '', text.strip(), flags=re.MULTILINE).strip()
    try:
        start, end = text.find('{'), text.rfind('}') + 1
        if start != -1 and end > start:
            return json.loads(text[start:end])
    except json.JSONDecodeError:
        pass
    return None


def call_gemini(sentence: str, sentence_id: str, model_id: str) -> dict:
    try:
        resp = client.models.generate_content(
            model=model_id,
            contents=sentence,
            config=types.GenerateContentConfig(
                system_instruction=SYSTEM_PROMPT,
                temperature=0.0,
                top_p=1.0,
                max_output_tokens=4096,
            )
        )
        logger.info(f"[{sentence_id}][{model_id}] finish: {resp.candidates[0].finish_reason}")

        content = resp.text
        parsed  = parse_response(content) if content else None
        return {"status": "success", **parsed} if parsed else {"status": "parse_failed", "raw": content}

    except Exception as e:
        logger.exception(f"[{sentence_id}][{model_id}] Exception")
        return {"status": "exception", "error": str(e)}


def extract_fields(errors):
    if not errors:
        return "", "", ""
    spans        = " | ".join(e.get("error_span", "")  for e in errors)
    categories   = " | ".join(e.get("category", "")    for e in errors)
    descriptions = " | ".join(e.get("description", "") for e in errors)
    return spans, categories, descriptions

In [ ]:
# ============================================================================
# CELL 3 — Run single model: gemini-3.1-flash-lite-preview (Resume-Safe)
# ============================================================================
import os, re as _re

# Redefine with thinking disabled
def call_gemini(sentence: str, sentence_id: str, model_id: str) -> dict:
    try:
        resp = client.models.generate_content(
            model=model_id,
            contents=sentence,
            config=types.GenerateContentConfig(
                system_instruction=SYSTEM_PROMPT,
                temperature=0.0,
                top_p=1.0,
                max_output_tokens=4096,
                thinking_config=types.ThinkingConfig(thinking_budget=0),
            )
        )
        logger.info(f"[{sentence_id}][{model_id}] finish: {resp.candidates[0].finish_reason}")
        content = resp.text
        parsed  = parse_response(content) if content else None
        return {"status": "success", **parsed} if parsed else {"status": "parse_failed", "raw": content}
    except Exception as e:
        logger.exception(f"[{sentence_id}][{model_id}] Exception")
        return {"status": "exception", "error": str(e)}

if "all_records" not in globals():
    all_records = {}

def _parse_retry_delay(error_str: str, default: float = 65.0) -> float:
    m = _re.search(r"'retryDelay':\s*'(\d+)s'", error_str)
    if m:
        return float(m.group(1)) + 3.0
    return default

def _is_daily_limit(error_str: str) -> bool:
    return "per day" in error_str.lower() or "quota" in error_str.lower()

##############################################################


TARGET_MODEL    = "gemini-3.1-flash-lite-preview"

#############################################################

output_path     = OUTPUT_PATHS[TARGET_MODEL]
failed_statuses = {"api_error", "parse_failed", "exception"}

df = pd.read_excel(INPUT_PATH, dtype=str).fillna("")
df = df[df["Incorrect Sentences"].str.strip() != ""].reset_index(drop=True)
logger.info(f"Loaded {len(df)} sentences")

if os.path.exists(output_path):
    existing_df = pd.read_excel(output_path, dtype=str).fillna("")
    records     = existing_df.to_dict(orient="records")
    done_sids   = {
        r["Sentence_ID"] for r in records
        if r.get("Status") not in failed_statuses and r.get("Status") != ""
    }
    logger.info(f"Resuming — {len(done_sids)} done, {len(df) - len(done_sids)} remaining.")
else:
    records, done_sids = [], set()
    logger.info("No existing file — starting fresh.")

sid_to_idx = {r["Sentence_ID"]: i for i, r in enumerate(records)}
skip_model = False

for _, row in df.iterrows():
    if skip_model:
        break

    sid, sentence = row["Sentence_ID"], row["Incorrect Sentences"]
    if sid in done_sids:
        continue

    for attempt in range(2):
        result = call_gemini(sentence, sid, TARGET_MODEL)

        if result.get("status") == "exception":
            err = result.get("error", "")

            if "404" in err or "NOT_FOUND" in err:
                logger.warning(f"[{TARGET_MODEL}] 404 — model not found. Stopping.")
                skip_model = True; break

            if "limit: 0" in err or _is_daily_limit(err):
                logger.warning(f"[{TARGET_MODEL}] Quota exhausted. Stopping.")
                skip_model = True; break

            if ("503" in err or "UNAVAILABLE" in err) and attempt == 0:
                logger.warning(f"[{sid}] 503 server overload. Waiting 45s then retrying...")
                time.sleep(45); continue

            if ("429" in err or "RESOURCE_EXHAUSTED" in err) and attempt == 0:
                wait = _parse_retry_delay(err)
                logger.warning(f"[{sid}] Rate limited. Waiting {wait:.0f}s then retrying...")
                time.sleep(wait); continue

        break

    if skip_model:
        break

    errors             = result.get("errors", [])
    spans, cats, descs = extract_fields(errors)

    new_record = {
        "Sentence_ID":         sid,
        "Incorrect Sentences": sentence,
        "Model_has_errors":    result.get("has_errors", ""),
        "Model_error_spans":   spans,
        "Model_categories":    cats,
        "Model_descriptions":  descs,
        "Model_corrected":     result.get("corrected_sentence", ""),
        "Status":              result.get("status", "")
    }

    if sid in sid_to_idx:
        records[sid_to_idx[sid]] = new_record
    else:
        sid_to_idx[sid] = len(records)
        records.append(new_record)

    print(f"\n[{sid}] {sentence}")
    print(json.dumps(result, indent=2, ensure_ascii=False))
    print("-" * 60)

    pd.DataFrame(records).to_excel(output_path, index=False)
    time.sleep(5)

all_records[TARGET_MODEL] = records
success = sum(1 for r in records if r.get("Status") == "success")
failed  = sum(1 for r in records if r.get("Status") in failed_statuses)
logger.info(f"[{TARGET_MODEL}] Done — {success} success, {failed} failed, {len(records)} total.")


[WL_09] ଇତିହାସରେ ଅନେକ କାହାଣୀର ପ୍ରମାଣ আছি ଏବଂ ତାହା ମଧ୍ୟ ବିଭିନ୍ନ ତଥ୍ୟରୁ।
{
  "status": "success",
  "has_errors": true,
  "errors": [
    {
      "error_span": "ଆଛି",
      "category": "Grammatical Errors",
      "description": "The verb 'ଆଛି' is a Bengali-influenced form or a dialectal variation. In standard Odia, the correct third-person singular/plural existential verb is 'ଅଛି' (singular) or 'ଅଛି' (plural/general). Given the context of 'ପ୍ରମାଣ' (evidence), 'ଅଛି' is the grammatically correct standard form."
    }
  ],
  "corrected_sentence": "ଇତିହାସରେ ଅନେକ କାହାଣୀର ପ୍ରମାଣ ଅଛି ଏବଂ ତାହା ମଧ୍ୟ ବିଭିନ୍ନ ତଥ୍ୟରୁ।"
}
------------------------------------------------------------

[WL_10] ଏହି ସମୟ ମଧ୍ୟରେ, ବୁଦ୍ଧ ଏବଂ ମହାବୀরଙ୍କ ଭଳି ମହାନ ପୁରୁଷମାନେ ଆସିଥିଲେ ଯେଉଁମାନେ କେବଳ ଜ୍ଞାନ ହାସଲ କରିନଥିଲେ ବରଂ ଏହାକୁ ଅନ୍ୟମାନଙ୍କ ପାଖରେ ମଧ୍ୟ ପହଞ୍ଚାଇଥିଲେ।
{
  "status": "success",
  "has_errors": false,
  "errors": [],
  "corrected_sentence": "ଏହି ସମୟ ମଧ୍ୟରେ, ବୁଦ୍ଧ ଏବଂ ମହାବୀରଙ୍କ ଭଳି ମହାନ ପୁରୁଷମାନେ ଆସିଥିଲେ ଯେଉଁମାନେ କେବଳ ଜ୍ଞାନ 

ERROR:__main__:[CS_07][gemini-3-flash-preview] Exception
Traceback (most recent call last):
  File "/tmp/ipykernel_21398/2543389125.py", line 9, in call_gemini
    resp = client.models.generate_content(
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/google/genai/models.py", line 5864, in generate_content
    response = self._generate_content(
               ^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/google/genai/models.py", line 4526, in _generate_content
    response = self._api_client.request(
               ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/google/genai/_api_client.py", line 1402, in request
    response = self._request(http_request, http_options, stream=False)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/google/genai/_api_client.py", line 1238, in _request
    return self._retry(self._request_once,


[CS_07] ଏହା ବ୍ୟତୀତ, ଏହା ମଧ୍ୟ ଦେଖାଯାଇଛି ଯେ ସାଧାରଣ ଲୋକମାନେ ଏ ବିଷୟରେ ସେତେ ଜାଗରୁକ ନାହାଁନ୍ତି।
{
  "status": "success",
  "has_errors": true,
  "errors": [
    {
      "error_span": "ନାହାଁନ୍ତି",
      "category": "Script Normalization",
      "description": "Incorrect placement of the chandrabindu (nasalization mark). In Odia, the chandrabindu should be placed on the vowel or the base consonant before the 'ti' suffix, or more commonly in this verb form, the anusvara is used. However, the specific encoding 'ନାହାଁନ୍ତି' contains a misplaced nasal mark relative to the 'ti' inflection. The correct standard form is 'ନାହାନ୍ତି'."
    }
  ],
  "corrected_sentence": "ଏହା ବ୍ୟତୀତ, ଏହା ମଧ୍ୟ ଦେଖାଯାଇଛି ଯେ ସାଧାରଣ ଲୋକମାନେ ଏ ବିଷୟରେ ସେତେ ଜାଗରୁକ ନାହାନ୍ତି।"
}
------------------------------------------------------------

[CS_08] ମହାଦେବ ସତୀଙ୍କୁ ଏତେ ଭଲପାଉଥିଲେ ଯେ, ସେ ଯଜ୍ଞକୁଣ୍ଡରୁ ଦଗ୍ଧ ଶରୀରକୁ ନିଜ କାନ୍ଧରେ ନେଇ ନାନା ଦେଶ ପାଗଳପ୍ରାୟ ଘୂରିବୁଲିଲେ। 
{
  "status": "success",
  "has_errors": true,
  "errors": [
    {
      "erro

In [ ]:
# ============================================================================
# CELL 3.1 — Retry failed rows for gemini-3.1-flash-lite-preview
# ============================================================================
failed_statuses = {"api_error", "parse_failed", "exception"}

out_df  = pd.read_excel(output_path, dtype=str).fillna("")
failed  = out_df[out_df["Status"].isin(failed_statuses)]

if failed.empty:
    print(f"[{TARGET_MODEL}] No failed rows.")
else:
    print(f"[{TARGET_MODEL}] Retrying {len(failed)} failed row(s)...")
    for idx, row in failed.iterrows():
        sid, sentence      = row["Sentence_ID"], row["Incorrect Sentences"]
        result             = call_gemini(sentence, sid, TARGET_MODEL)
        errors             = result.get("errors", [])
        spans, cats, descs = extract_fields(errors)
        out_df.at[idx, "Model_has_errors"]    = result.get("has_errors", "")
        out_df.at[idx, "Model_error_spans"]   = spans
        out_df.at[idx, "Model_categories"]    = cats
        out_df.at[idx, "Model_descriptions"]  = descs
        out_df.at[idx, "Model_corrected"]     = result.get("corrected_sentence", "")
        out_df.at[idx, "Status"]              = result.get("status", "")
        print(f"  [{sid}] {result.get('status')}")
        time.sleep(4)

    out_df.to_excel(output_path, index=False)
    print(f"[{TARGET_MODEL}] Saved.")

[gemini-3-flash-preview] No failed rows.
